<a href="https://colab.research.google.com/github/gaurinandwana/calc/blob/main/task1_GenerativeAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets

In [4]:
# Install libraries
!pip install transformers datasets

# Imports
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
from datasets import Dataset
import torch

# -------------------------------
# STEP 1: Dataset (your own text)
# -------------------------------
data = [
"It is really hard to build yourself up when all you do is constantly tear yourself down",
"You are the only person you need to be good enough for.",
"Be somebody who makes everybody feel like a somebody.",
"Believe in yourself even when no one else does.",
"The main reason I want more is so I can give more.",
"Consistency is more powerful than motivation.",
"And that is when I realised you're not that great.",
"Life goes on.",
"Push yourself beyond your limits.",
"Great things take time and patience.",
"Your aura gets prettier every time you heal.",
"May we endure less and enjoy more."
]

# -------------------------------
# STEP 2: Load model + tokenizer
# -------------------------------
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token   # Fix padding issue

model = GPT2LMHeadModel.from_pretrained("gpt2")

# -------------------------------
# STEP 3: BEFORE TRAINING OUTPUT
# -------------------------------
print("===== BEFORE TRAINING =====\n")

prompt = "Life is"
inputs = tokenizer.encode(prompt, return_tensors="pt")

outputs = model.generate(inputs, max_length=40)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


# -------------------------------
# STEP 4: Prepare dataset
# -------------------------------
dataset = Dataset.from_dict({"text": data})

def tokenize(example):
    encoding = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=50
    )
    encoding["labels"] = encoding["input_ids"]   # Important for training
    return encoding

tokenized_dataset = dataset.map(tokenize, remove_columns=["text"])


# -------------------------------
# STEP 5: Training setup
# -------------------------------
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    logging_steps=5,
    save_steps=10,
    save_total_limit=1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# -------------------------------
# STEP 6: Train model
# -------------------------------
trainer.train()


# -------------------------------
# STEP 7: AFTER TRAINING OUTPUT
# -------------------------------
print("\n===== AFTER TRAINING =====\n")

outputs = model.generate(
    inputs,
    max_length=40,
    num_return_sequences=3,
    do_sample=True,
    top_k=50,
    top_p=0.95
)

for i, output in enumerate(outputs):
    print(f"\nOutput {i+1}:")
    print(tokenizer.decode(output, skip_special_tokens=True))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


===== BEFORE TRAINING =====

Life is a great way to get to know your friends and family.

You can also find out more about the best places to meet people in your area.

If you're looking for


Map:   0%|          | 0/12 [00:00<?, ? examples/s]

Step,Training Loss
5,4.429510
10,0.732078
15,0.585452
20,0.653331
25,0.358465
30,0.465612


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



===== AFTER TRAINING =====


Output 1:
Life is a complex thing.

Output 2:
Life is just like a whole bunch of stuff you're only so much better on at

Output 3:
Life is the only way to get there
